# Predicción de producción de pozos petroleros para OilyGiant.


# Objetivo
Identificar la región con el mayor beneficio esperado y el menor riesgo financiero para abrir 200 nuevos pozos petroleros.
Para lograrlo, se entrenará un modelo de regresión lineal que prediga el volumen de reservas en cada sitio, se seleccionarán los 200 pozos potencialmente más productivos por región, y se evaluarán sus ganancias (y riesgos) mediante simulaciones de bootstrapping.

## Importar Librerías

In [1]:
# Librerías básicas
import pandas as pd
import numpy as np

# Para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Para preprocesamiento 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error


## Carga y previsualización de los datos

In [2]:
geo_0 = pd.read_csv('/Users/irmalupita/Desktop/Proyecto11/geo_data_0.csv')

geo_1 = pd.read_csv('/Users/irmalupita/Desktop/Proyecto11/geo_data_1.csv')

geo_2 = pd.read_csv('/Users/irmalupita/Desktop/Proyecto11/geo_data_2.csv')

geo_0.info()
geo_0.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,count,mean,std,min,25%,50%,75%,max
f0,100000.0,0.500419,0.871832,-1.408605,-0.072580,0.502360,1.073581,2.362331
f1,100000.0,0.250143,0.504433,-0.848218,-0.200881,0.250252,0.700646,1.343769
f2,100000.0,2.502647,3.248248,-12.088328,0.287748,2.515969,4.715088,16.003790
product,100000.0,92.500000,44.288691,0.000000,56.497507,91.849972,128.564089,185.364347


In [3]:
geo_1.info()
geo_1.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,count,mean,std,min,25%,50%,75%,max
f0,100000.0,1.141296,8.965932,-31.609576,-6.298551,1.153055,8.621015,29.421755
f1,100000.0,-4.796579,5.119872,-26.358598,-8.267985,-4.813172,-1.332816,18.734063
f2,100000.0,2.494541,1.703572,-0.018144,1.000021,2.011479,3.999904,5.019721
product,100000.0,68.825000,45.944423,0.000000,26.953261,57.085625,107.813044,137.945408


In [4]:
geo_2.info()
geo_2.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


,count,mean,std,min,25%,50%,75%,max
f0,100000.0,0.002023,1.732045,-8.760004,-1.162288,0.009424,1.158535,7.238262
f1,100000.0,-0.002081,1.730417,-7.084020,-1.174820,-0.009482,1.163678,7.844801
f2,100000.0,2.495128,3.473445,-11.970335,0.130359,2.484236,4.858794,16.739402
product,100000.0,95.000000,44.749921,0.000000,59.450441,94.925613,130.595027,190.029838


In [5]:
# Revisamos duplicados
print("Duplicados en geo_0:", geo_0.duplicated().sum())
print("Duplicados en geo_1:", geo_1.duplicated().sum())
print("Duplicados en geo_2:", geo_2.duplicated().sum())

Duplicados en geo_0: 0
Duplicados en geo_1: 0
Duplicados en geo_2: 0


**Observaciones:**
Al revisar los tres archivos, vemos que los datos vienen en buen estado y no requieren una limpieza profunda. No aparecen filas duplicadas y cada columna tiene el tipo correcto de dato: las características (f0, f1, f2) y la variable objetivo ('product') son numéricas, mientras que el identificador del pozo es texto. Esto nos indica que el dataset no tiene inconsistencias básicas y que se puede trabajar directamente con él.

Los valores extremos que aparecen no se consideran errores pues no vemos valores imposibles como números exageradamente grandes o negativos fuera de contexto. Los rangos amplios también se ven equilibrados entre el mínimo y el máximo respecto a la media.

El análisis con describe() permite entender mejor cómo se comporta cada región. En geo_0, las desviaciones estándar de las características son bajas; esto podría significar que los datos están más concentrados y estables, facilitando que un modelo de regresión encuentre patrones. La media de 'product' es de unos 92.5, lo que indica una producción relativamente alta.

En cambio, geo_1 muestra la producción media más baja, alrededor de 68.8, lo que hace que esta región sea menos atractiva desde el punto de vista económico.

Finalmente, la región geo_2 presenta una dispersión moderada: más amplia que en geo_0 pero mucho más controlada que en geo_1. Aquí, la producción promedio es la más alta, cercana a 95, lo que convierte a esta región en una candidata fuerte desde la perspectiva del beneficio potencial.

## Entrenamiento de modelo por región

In [ ]:
## Separamos el objetivo de las variables
#X = geo_0.drop(columns=['product', 'id'])
#y = geo_0['product']

## Dividir en train y validación (75:25)
#X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42)

## Entrenamiento
#model = LinearRegression()
#model.fit(X_train, y_train)

## Predicción
#preds = model.predict(X_valid)

## Guardar predicciones y respuestas reales
#results_geo0 = y_valid.to_frame(name='actual')
#results_geo0['predicted'] = preds
#results_geo0.head()

## Calcular volumen medio y RMSE (forma nueva)
#mean_pred = preds.mean()
#rmse = root_mean_squared_error(y_valid, preds)

#mean_pred, rmse

(92.3987999065777, 37.75660035026169)

A continuación se entrena un modelo apra predecir cuántas unidades de petróleo puede generar un pozo nuevo en cada región.

¿Por qué necesitamos predecir eso?
Porque no sabemos todavía cuáles pozos reales producirán más. Pero sí tenemos datos geológicos (“f0”, “f1”, “f2”) de cada pozo candidato. Así que entrenamos un modelo para que aprenda esta relación:

Datos geológicos --> producción esperada del pozo

Con esas predicciones decidiremos:

- Cuáles son los 200 pozos más prometedores en cada región.
- Cuánto dinero podríamos ganar si perforamos esos pozos.
- Qué región tiene menos riesgo y más beneficio.

In [10]:
## Función para entrenar modelo por región
def entrenar_model(df):
    X = df.drop(columns=['product', 'id'])
    y = df['product']
    
    X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42)
    
    model = LinearRegression()
    
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    mean_pred = preds.mean()
    rmse = root_mean_squared_error(y_valid, preds)

    return model, preds, y_valid.reset_index(drop=True), mean_pred, rmse

In [13]:
model_0, pred_0, valid_0, mean_pread_0, rmse_0 = entrenar_model(geo_0)
model_1, pred_1, valid_1, mean_pread_1, rmse_1 = entrenar_model(geo_1)
model_2, pred_2, valid_2, mean_pread_2, rmse_2 = entrenar_model(geo_2)

print("Mean & RMSE Región 0:", mean_pread_0, rmse_0)
print("Mean & RMSE Región 1:", mean_pread_1, rmse_1)
print("Mean & RMSE Región 2:", mean_pread_2, rmse_2)

Mean & RMSE Región 0: 92.3987999065777 37.75660035026169
Mean & RMSE Región 1: 68.71287803913762 0.8902801001028817
Mean & RMSE Región 2: 94.7710238776594 40.145872311342174


In [14]:
pred_0[:10]

array([101.90101715,  78.21777385, 115.26690103, 105.61861791,
        97.9801849 ,  80.91536742,  81.48585214,  94.66397503,
        81.52637566,  99.38973974])

In [15]:
pred_1[:10]

array([ 8.44738063e-01,  5.29216119e+01,  1.35110385e+02,  1.09494863e+02,
       -4.72915824e-02,  1.36356564e+02,  2.66088928e+00,  2.67962114e+01,
        3.03183219e+01,  8.17331406e+01])

In [ ]:
[101.90101715,  78.21777385, 115.26690103, 105.61861791,
        97.9801849 ,  80.91536742,  81.48585214,  94.66397503,
        81.52637566,  99.38973974])

In [16]:
pred_2[:10]

array([ 98.30191642, 101.59246124,  52.4490989 , 109.92212707,
        72.41184733, 104.14254172,  64.12765636,  91.91858202,
       104.93798646,  92.21848489])

Conclusión

Al comparar las tres regiones, vemos que geo_1, aunque obtiene el RMSE más bajo, no es precisamente una buena opción para invertir porque su producción promedio es la más baja. La buena precisión del modelo no la vuelve más rentable.

Las regiones geo_0 y geo_2 sí muestran producciones más altas, pero también errores mayores en la predicción. Aun así, geo_2 destaca porque combina el promedio más alto de producción con un error aceptable, lo que la convierte en la candidata más fuerte para el análisis económico posterior.

## Calcular ganancia
Se realiza una función para calcular las ganacias de los 200 pozos
para ellos se seleccionarn lo 200 mejores pozos predichos por el modelo  y se comprueba la ganancia con base en los valores reales.

In [ ]:
# Parámetros económicos (guardado en variables)
presupuesto = 100_000_000          # USD (100 millones)
n_pozos = 200                      # pozos a perforar
costo_por_pozo = presupuesto / n_pozos   # 500000 USD
ingreso_por_unidad = 4500          # USD por unidad (product está en miles de barriles)


In [21]:
## Función para calcular ganancia

def calcular_ganancia (preds, y_valid):
    indices_top = preds.sort_values(ascending=False).index[:n_pozos]
    ganancia = y_valid[indices_top].sum() * ingreso_por_unidad - presupuesto
    return ganancia

gan_0 = calcular_ganancia(pd.Series(pred_0), valid_0)
gan_1 = calcular_ganancia(pd.Series(pred_1), valid_1)
gan_2 = calcular_ganancia(pd.Series(pred_2), valid_2)

print ("Ganancia Región 0:", gan_0)
print ("Ganancia Región 1:", gan_1)
print ("Ganancia Región 2:", gan_2)

Ganancia Región 0: 33591411.14462179
Ganancia Región 1: 24150866.966815114
Ganancia Región 2: 25985717.59374112


**Comentario:**
Las ganancias iniciales muestran que la Región 0 parece la más rentable, pero su RMSE es muy alto, lo que indica que el modelo se equivoca bastante al predecir. Eso hace que sus ganancias sean más inciertas. La Región 1 tiene la menor ganancia estimada, pero también el RMSE más bajo, lo que la vuelve la región más estable y confiable. La Región 2 queda en un punto medio tanto en ganancia como en precisión.

En este punto, la Región 1 parece la más segura, mientras que la Región 0 es más arriesgada aunque muestre ganancias altas.

Para decidir de forma más sólida, usamos bootstrapping. Una sola estimación de ganancia no basta porque depende de una división específica del dataset y no siempre refleja lo que pasaría en la práctica. El bootstrapping simula miles de escenarios posibles y nos da una distribución completa del beneficio: cuánto puede variar, cuál es la probabilidad de perder dinero y cuál es el intervalo de confianza. Así, la elección final se basa no solo en cuánto podría ganar cada región, sino en qué tan estable es ese resultado.

In [ ]:
def ganancia_bootstrap(preds, y_valid):
    ganancia = [] 
    for _ in range(1000):
        sample = pd.Series(np.random.choice(real, size=500, replace=True))
        indices_top = sample.sort_values(ascending=False).index[:n_pozos]
        ganacias.append(real_sample[top_indices].sum()* ingreso_por_unidad - presupuesto)
    return ganancias

In [ ]:
np.random.seed(42)

def ganancia_bootstrap(pred, real):
    ganancias = []
    for _ in range(1000):
        # Se toman 500 predicciones y valores reales con reemplazo
        sample = pd.Series(np.random.choice(pred, size=500, replace=True))
        real_sample = pd.Series(np.random.choice(real, size=500, replace=True))

        # Se seleccionan los índices de las mejores predicciones
        top_indices = sample.sort_values(ascending=False).index[:n_pozos]

        # Se calcula la ganancia real de esos pozos
        ganancias.append(real_sample[top_indices].sum() * ingreso_por_unidad - presupuesto)
    
    return ganancias


def analizar_riesgo(ganancias):
    ganancias = pd.Series(ganancias)
    media = ganancias.mean()
    intervalo = ganancias.quantile([0.025, 0.975])
    riesgo = (ganancias < 0).mean() * 100
    return media, intervalo, riesgo


# Bootstrapping de cada región
gan_boot_0 = ganancia_bootstrap(pred_0, valid_0)
gan_boot_1 = ganancia_bootstrap(pred_1, valid_1)
gan_boot_2 = ganancia_bootstrap(pred_2, valid_2)

# Resultados
for i, g in enumerate([gan_boot_0, gan_boot_1, gan_boot_2]):
    media, intervalo, riesgo = analizar_riesgo(g)
    print(f"Región {i} -> Ganancia Prom: {media:.2f}, IC 95%: {intervalo.values}, Riesgo pérdida: {riesgo:.2f}%")


Región 0 -> Ganancia Prom: -17032724.60, IC 95%: [-22532131.77760003 -11815080.67529193], Riesgo pérdida: 100.00%
Región 1 -> Ganancia Prom: -38120428.82, IC 95%: [-43511526.57964028 -32673298.95305358], Riesgo pérdida: 100.00%
Región 2 -> Ganancia Prom: -14536834.96, IC 95%: [-19770087.21728544  -8752812.770656  ], Riesgo pérdida: 100.00%


**Conclusión**
Los resultados de las 1000 simulaciones por región muestran de manera consistente que ninguna de las tres zonas presenta rentabilidad: todas las distribuciones de ganancia se mantienen en valores negativos, con un riesgo de pérdida del 100% y con intervalos de confianza al 95% que no incluyen ningún escenario positivo.

Si bien la Región 2 muestra la menor pérdida promedio dentro del conjunto, continúa operando en márgenes negativos. La Región 1 presenta el comportamiento más desfavorable, mientras que la Región 0 tampoco ofrece una perspectiva viable de recuperación o beneficio.

Estos hallazgos contrastan con las conclusiones preliminares obtenidas a partir del análisis de la ganancia estimada y del RMSE, donde algunas regiones parecían más prometedoras. El uso de bootstrapping aporta una visión más robusta y revela que la aparente estabilidad de los modelos no se traduce en ganancias reales cuando se considera la variabilidad estadística.

En consecuencia, con la evidencia generada, no se recomienda invertir en el desarrollo de pozos petrolíferos en ninguna de las regiones evaluadas, ya que ninguna de ellas ofrece una probabilidad realista de obtener beneficios económicos bajo las condiciones actuales.